# Crop Advisor Lite — AMD Compute Evidence

**Purpose:** demonstrate, visibly and reproducibly, that this project's workflow runs on
**AMD Instinct GPU compute** via the AMD Developer Cloud.

Run this notebook **on an AMD Developer Cloud GPU pod** (AMD Hackathon: ACT II). It:

1. Proves the AMD hardware (`rocm-smi`, the AMD driver stack).
2. Confirms PyTorch sees the AMD GPU through ROCm.
3. Runs a real GPU computation (timed matrix multiply) on the AMD Instinct GPU.
4. Runs the **retrieval-embedding step of Crop Advisor Lite's RAG pipeline on the AMD GPU** —
   embedding the project's 40-document Pakistani-agriculture corpus with an open-weight
   embedding model. This is the project's real next step (BM25 → dense embeddings), executed
   on AMD silicon.

Together with the app's inference path — `gpt-oss-120b` served on AMD via the Fireworks AI
API — **the whole Crop Advisor Lite pipeline runs on AMD compute.**

## 1. AMD hardware proof — `rocm-smi`

This lists the AMD Instinct GPU(s) on this pod. ROCm is AMD's GPU compute stack (their CUDA equivalent).

In [ ]:
!rocm-smi

## 2. PyTorch sees the AMD GPU (via ROCm)

PyTorch's ROCm build exposes AMD GPUs through the same `torch.cuda` API. The device name below is the AMD Instinct part.

In [ ]:
import torch

print("PyTorch version :", torch.__version__)
print("ROCm/HIP version:", getattr(torch.version, "hip", None))
print("GPU available   :", torch.cuda.is_available())
print("GPU count       :", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU 0 name      :", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "No GPU visible — run this on an AMD Developer Cloud GPU pod."


## 3. Real computation on the AMD GPU

A large matrix multiply executed on the AMD Instinct GPU, timed. This is actual FLOPs on AMD silicon, no internet required.

In [ ]:
import torch, time

device = "cuda"  # ROCm maps AMD GPUs to the cuda device
n = 8192
a = torch.randn(n, n, device=device)
b = torch.randn(n, n, device=device)

torch.cuda.synchronize()
start = time.time()
for _ in range(10):
    c = a @ b
torch.cuda.synchronize()
elapsed = time.time() - start

flops = 10 * 2 * n**3
print(f"Matrix size      : {n} x {n}")
print(f"10 matmuls in    : {elapsed:.3f} s on {torch.cuda.get_device_name(0)}")
print(f"Throughput       : {flops/elapsed/1e12:.1f} TFLOP/s")
print(f"Result checksum  : {c.sum().item():.2f}  (proves the compute ran)")


## 4. Project compute on AMD — embed the Crop Advisor Lite corpus

Crop Advisor Lite currently retrieves with BM25 (keyword search). The planned upgrade is
**dense embedding retrieval** so questions match documents by meaning, not just shared words.
Here that upgrade runs **on the AMD GPU**: we load an open-weight embedding model onto the
AMD Instinct GPU and embed the project's 40-document corpus, then run a semantic search.

The corpus is read from `../corpus/` (present when this notebook is run inside a clone of
the repo). If you opened the notebook standalone, the next cell git-clones the public repo.

In [ ]:
import os, glob, re, subprocess

# Locate the corpus. If running inside the cloned repo, it's ../corpus.
corpus_dir = None
for cand in ["../corpus", "corpus", "crop-advisor-lite/corpus"]:
    if os.path.isdir(cand):
        corpus_dir = cand
        break

if corpus_dir is None:
    print("Corpus not found locally — cloning the public repo...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/koco9people/crop-advisor-lite.git"], check=True)
    corpus_dir = "crop-advisor-lite/corpus"

print("Using corpus at:", corpus_dir)

def load_chunks(corpus_dir, chunk_words=400, stride=320):
    header_re = re.compile(r"<!--.*?-->", re.DOTALL)
    chunks = []
    for path in sorted(glob.glob(os.path.join(corpus_dir, "*.md"))):
        if os.path.basename(path) == "SOURCES.md":
            continue
        text = header_re.sub("", open(path).read())
        title = os.path.basename(path).replace(".md", "")
        words = text.split()
        for start in range(0, max(len(words) - chunk_words // 2, 1), stride):
            passage = " ".join(words[start:start + chunk_words])
            if len(passage.split()) >= 40:
                chunks.append({"title": title, "text": passage})
    return chunks

chunks = load_chunks(corpus_dir)
print(f"Loaded {len(chunks)} chunks from {len(set(c['title'] for c in chunks))} documents")


In [ ]:
# Install the embedding library (needs outbound internet on the pod)
!pip -q install sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer
import torch, time

# Load an open-weight embedding model ONTO THE AMD GPU
model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
print("Embedding model loaded on:", torch.cuda.get_device_name(0))

texts = [c["text"] for c in chunks]
start = time.time()
embeddings = model.encode(texts, batch_size=64, convert_to_tensor=True,
                          device="cuda", show_progress_bar=True)
elapsed = time.time() - start
print(f"Embedded {len(texts)} corpus chunks on the AMD GPU in {elapsed:.1f}s")
print("Embedding tensor:", tuple(embeddings.shape), "on device", embeddings.device)


### Semantic search demo — the AMD-computed embeddings answering a real query

In [ ]:
query = "When should I sow cotton in southern Punjab?"
q_emb = model.encode(query, convert_to_tensor=True, device="cuda")

scores = torch.nn.functional.cosine_similarity(q_emb.unsqueeze(0), embeddings)
top = torch.topk(scores, 3)

print("Query:", query, "\n")
for rank, (score, idx) in enumerate(zip(top.values.tolist(), top.indices.tolist()), 1):
    c = chunks[idx]
    print(f"{rank}. [{score:.3f}] {c['title']}")
    print("   ", c["text"][:160], "...\n")


## Summary — AMD compute in Crop Advisor Lite

| Stage | Runs on | Shown here |
|-------|---------|------------|
| Retrieval embeddings (planned upgrade) | **AMD Instinct GPU** (this notebook, AMD Developer Cloud) | ✅ `rocm-smi` + on-GPU matmul + corpus embedding |
| Inference (`gpt-oss-120b`) | **AMD** via Fireworks AI API | app's live default path |

Every model computation in this project — the embeddings computed above and the LLM
inference served by Fireworks — runs on AMD hardware. This notebook is the direct,
reproducible evidence of AMD GPU usage in the workflow.